<a href="https://colab.research.google.com/github/Kishan-kp-ai/Linkific-Tasks/blob/main/Day%2016/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


# ============================================================
# 1. INSTALL ONLY REQUIRED PACKAGES
# ============================================================

!pip -q install -U sentence-transformers faiss-cpu transformers


# ============================================================
# 2. IMPORTS
# ============================================================

import time
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


print("=" * 60)
print("BASIC RAG PROJECT")
print("=" * 60)


# ============================================================
# 3. DOCUMENTS
# ============================================================

documents = [

    {
        "source": "machine_learning.txt",
        "text": """
Machine learning is a branch of artificial intelligence that allows
computers to learn patterns from data and make predictions without
being explicitly programmed.

The three main types of machine learning are supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labeled data. Unsupervised learning works
with unlabeled data. Reinforcement learning allows an agent to learn
through rewards and penalties.
"""
    },

    {
        "source": "nlp.txt",
        "text": """
Natural Language Processing, also known as NLP, is a field of
artificial intelligence that helps computers understand and process
human language.

NLP is used for sentiment analysis, text classification, translation,
question answering, text summarization, and text generation.
"""
    },

    {
        "source": "rag.txt",
        "text": """
Retrieval-Augmented Generation, commonly called RAG, combines
information retrieval with a language model.

In a RAG system, documents are divided into smaller chunks.
Each chunk is converted into a numerical vector called an embedding.

The embeddings are stored in a vector database. When a user asks
a question, the question is converted into an embedding and the
system searches for similar document chunks.

The retrieved chunks are then provided to a language model as
context. The language model uses this context to generate an answer.

RAG helps language models provide answers based on a specific
collection of documents.
"""
    }
]


print("\nDocuments loaded:", len(documents))

for doc in documents:
    print(
        "-",
        doc["source"],
        "|",
        len(doc["text"]),
        "characters"
    )


# ============================================================
# 4. LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")


# ============================================================
# 5. CREATE CHUNKS
# ============================================================

def create_chunks(documents, chunk_size):

    chunks = []

    for document in documents:

        text = document["text"].strip()

        for start in range(
            0,
            len(text),
            chunk_size
        ):

            chunk = text[
                start:start + chunk_size
            ].strip()

            if chunk:

                chunks.append({
                    "text": chunk,
                    "source": document["source"]
                })

    return chunks


# ============================================================
# 6. CREATE FAISS VECTOR DATABASE
# ============================================================

def create_vector_database(chunks):

    texts = [
        chunk["text"]
        for chunk in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = embeddings.astype(
        "float32"
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(
        embeddings
    )

    return index


# ============================================================
# 7. RETRIEVE RELEVANT DOCUMENTS
# ============================================================

def retrieve(
    question,
    chunks,
    index,
    top_k=2
):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_embedding = question_embedding.astype(
        "float32"
    )

    scores, indices = index.search(
        question_embedding,
        top_k
    )

    retrieved = []

    for index_number, score in zip(
        indices[0],
        scores[0]
    ):

        if index_number != -1:

            retrieved.append({
                "text": chunks[index_number]["text"],
                "source": chunks[index_number]["source"],
                "score": float(score)
            })

    return retrieved


# ============================================================
# 8. LOAD FLAN-T5 SMALL
# ============================================================

print("\nLoading language model...")

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Language model loaded successfully.")


# ============================================================
# 9. LLM GENERATION FUNCTION
# ============================================================

def generate_answer(
    question,
    retrieved_chunks
):

    context = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks
    )

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():

        output = llm.generate(
            **inputs,
            max_new_tokens=80
        )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer


# ============================================================
# 10. BASIC RAG TEST
# ============================================================

question = "What is Retrieval-Augmented Generation?"

print("\n")
print("=" * 60)
print("BASIC RAG TEST")
print("=" * 60)

chunks = create_chunks(
    documents,
    chunk_size=400
)

index = create_vector_database(
    chunks
)

retrieved = retrieve(
    question,
    chunks,
    index,
    top_k=2
)

answer = generate_answer(
    question,
    retrieved
)


print("\nQuestion:")
print(question)

print("\nRetrieved Documents:")

for i, chunk in enumerate(
    retrieved,
    1
):

    print(
        f"\nDocument {i}:"
    )

    print(
        "Source:",
        chunk["source"]
    )

    print(
        "Similarity:",
        round(
            chunk["score"],
            3
        )
    )

    print(
        "Content:",
        chunk["text"]
    )


print("\nGenerated Answer:")
print(answer)


# ============================================================
# 11. CHUNK SIZE EXPERIMENT
# ============================================================

print("\n")
print("=" * 60)
print("CHUNK SIZE EXPERIMENT")
print("=" * 60)

# Only three sizes to keep the experiment fast
chunk_sizes = [
    200,
    400,
    600
]

results = []


for chunk_size in chunk_sizes:

    print(
        "\nTesting chunk size:",
        chunk_size
    )

    start_time = time.time()

    # Create chunks
    chunks = create_chunks(
        documents,
        chunk_size
    )

    # Create vector database
    index = create_vector_database(
        chunks
    )

    # Retrieve relevant chunks
    retrieved = retrieve(
        question,
        chunks,
        index,
        top_k=2
    )

    # Generate answer
    answer = generate_answer(
        question,
        retrieved
    )

    execution_time = (
        time.time() -
        start_time
    )

    # Calculate average similarity
    similarity = np.mean(
        [
            item["score"]
            for item in retrieved
        ]
    )

    results.append({
        "Chunk Size": chunk_size,
        "Number of Chunks": len(chunks),
        "Average Similarity": round(
            float(similarity),
            3
        ),
        "Execution Time (seconds)": round(
            execution_time,
            2
        ),
        "Generated Answer": answer
    })

    print(
        "Number of chunks:",
        len(chunks)
    )

    print(
        "Average similarity:",
        round(
            float(similarity),
            3
        )
    )

    print(
        "Execution time:",
        round(
            execution_time,
            2
        ),
        "seconds"
    )

    print(
        "Answer:",
        answer
    )


# ============================================================
# 12. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 60)
print("RAG PERFORMANCE RESULTS")
print("=" * 60)

display(
    results_df
)


# ============================================================
# 13. FIND BEST RETRIEVAL SCORE
# ============================================================

best_row = results_df.loc[
    results_df["Average Similarity"].idxmax()
]

best_chunk_size = int(
    best_row["Chunk Size"]
)

best_similarity = float(
    best_row["Average Similarity"]
)


# ============================================================
# 14. FINAL ANALYSIS
# ============================================================

print("\n")
print("=" * 60)
print("RAG RETRIEVAL PERFORMANCE ANALYSIS")
print("=" * 60)

print(
    f"""
The experiment tested three different chunk sizes:

200 characters
400 characters
600 characters

The system measured:

1. Number of chunks
2. Embedding and retrieval performance
3. Similarity between the question and retrieved chunks
4. LLM response
5. Execution time

Smaller chunks:
- Create more chunks.
- Provide more focused information.
- May lose surrounding context.

Larger chunks:
- Create fewer chunks.
- Preserve more context.
- May contain unnecessary information.

In this experiment, the highest measured retrieval
similarity was obtained with:

Chunk size: {best_chunk_size}

Average similarity: {best_similarity}

This result applies only to this small experimental
dataset and should not be considered a universal
best chunk size.
"""
)


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("COMPLETE RAG PIPELINE")
print("=" * 60)

print(
    """
Documents
    ↓
Document Chunking
    ↓
Sentence Transformer Embeddings
    ↓
FAISS Vector Database
    ↓
Similarity Search
    ↓
Relevant Document Retrieval
    ↓
FLAN-T5 LLM
    ↓
Generated RAG Answer
    ↓
Chunk Size Performance Analysis
"""
)

print("=" * 60)
print("TASK COMPLETED SUCCESSFULLY")
print("=" * 60)

BASIC RAG PROJECT

Documents loaded: 3
- machine_learning.txt | 451 characters
- nlp.txt | 279 characters
- rag.txt | 633 characters

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.

Loading language model...


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Language model loaded successfully.


BASIC RAG TEST

Question:
What is Retrieval-Augmented Generation?

Retrieved Documents:

Document 1:
Source: rag.txt
Similarity: 0.619
Content: Retrieval-Augmented Generation, commonly called RAG, combines
information retrieval with a language model.

In a RAG system, documents are divided into smaller chunks.
Each chunk is converted into a numerical vector called an embedding.

The embeddings are stored in a vector database. When a user asks
a question, the question is converted into an embedding and the
system searches for similar docum

Document 2:
Source: rag.txt
Similarity: 0.361
Content: ent chunks.

The retrieved chunks are then provided to a language model as
context. The language model uses this context to generate an answer.

RAG helps language models provide answers based on a specific
collection of documents.

Generated Answer:
RAG


CHUNK SIZE EXPERIMENT

Testing chunk size: 200
Number of chunks: 9
Average similarity: 0.517
Execution t

,Chunk Size,Number of Chunks,Average Similarity,Execution Time (seconds),Generated Answer
0,200,9,0.517,1.14,combines information retrieval with a language...
1,400,5,0.490,1.81,RAG
2,600,4,0.463,3.97,combines information retrieval with a language...




RAG RETRIEVAL PERFORMANCE ANALYSIS

The experiment tested three different chunk sizes:

200 characters
400 characters
600 characters

The system measured:

1. Number of chunks
2. Embedding and retrieval performance
3. Similarity between the question and retrieved chunks
4. LLM response
5. Execution time

Smaller chunks:
- Create more chunks.
- Provide more focused information.
- May lose surrounding context.

Larger chunks:
- Create fewer chunks.
- Preserve more context.
- May contain unnecessary information.

In this experiment, the highest measured retrieval
similarity was obtained with:

Chunk size: 200

Average similarity: 0.517

This result applies only to this small experimental
dataset and should not be considered a universal
best chunk size.



COMPLETE RAG PIPELINE

Documents
    ↓
Document Chunking
    ↓
Sentence Transformer Embeddings
    ↓
FAISS Vector Database
    ↓
Similarity Search
    ↓
Relevant Document Retrieval
    ↓
FLAN-T5 LLM
    ↓
Generated RAG Answer
    ↓
Chu